# BirdCLEF 2026 — improved CPU/GPU training pipeline

This notebook updates the original baseline with several performance-oriented improvements:

1. Consistent configuration and experiment naming.
2. Random multi-chunk training instead of only the loudest 5 s crop.
3. Validation with deterministic multi-crop aggregation.
4. Optional fold-style split seed rotation support.
5. Smaller, explicit default backbone for faster training.
6. Optional backbone freezing warmup.
7. Better checkpoint metadata and cleaner inference aggregation.
8. Support for 1-channel spectrogram input or 3-channel ImageNet-style input.

The notebook is still designed to run on CPU, but benefits from GPU if available.


In [1]:
import os
import gc
import math
import json
import time
import random
import warnings
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchaudio
import librosa
import timm

from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

cpu_threads = max(1, (os.cpu_count() or 2) // 2)
torch.set_num_threads(cpu_threads)
torch.set_float32_matmul_precision('high') if hasattr(torch, 'set_float32_matmul_precision') else None

print('Torch:', torch.__version__)
print('CPU threads:', cpu_threads)


e:\OneDrive\Pulpit\BirdCLEF-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.12.0+cpu
CPU threads: 4


In [19]:
import kagglehub
kagglehub.dataset_download('wiktorwoniak/timm-weights')

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

100%|██████████| 15.3M/15.3M [00:01<00:00, 13.8MB/s]

Extracting files...


In [20]:
@dataclass
class Config:
    data_dir: str = r'data\birdclef-2026'
    cache_dir: str = 'mel_cache_v2'
    model_dir: str = 'models'
    model_weights_dir: str = '/kaggle/input/datasets/wiktorwoniak/timm-weights/effnet_b0_ns_jft_in1k.pth'

    sr: int = 32000
    chunk_seconds: int = 5
    n_fft: int = 1024
    hop_length: int = 512
    n_mels: int = 128
    fmin: int = 50
    fmax: int = 14000
    top_db: int = 80

    max_files_per_species: int = 40
    val_frac: float = 0.15
    test_frac: float = 0.15
    split_seed: int = 42

    batch_size: int = 24
    epochs: int = 4
    lr: float = 3e-4
    weight_decay: float = 1e-4
    mixup_alpha: float = 0.3
    use_spec_augment: bool = True
    secondary_label_weight: float = 0.25
    label_smoothing: float = 0.02
    freeze_backbone_epochs: int = 1
    grad_clip: float = 1.0

    model_name: str = 'tf_efficientnet_b0.ns_jft_in1k'
    pretrained: bool = True
    in_chans: int = 3
    dropout: float = 0.0

    train_random_chunks: int = 1
    val_num_chunks: int = 3
    infer_num_chunks: int = 12
    max_audio_seconds_for_cache_probe: int = 180

    sample_strategy: str = 'sqrt_inv'
    num_workers: int = 0
    amp: bool = True

    experiment_name: str = 'birdclef_improved_v1'

cfg = Config()
DEVICE = torch.device('cpu')
CHUNK_SAMPLES = cfg.sr * cfg.chunk_seconds
DATA_DIR = Path(cfg.data_dir)
CACHE_DIR = Path(cfg.cache_dir)
MODEL_DIR = Path(cfg.model_dir)
CACHE_DIR.mkdir(exist_ok=True, parents=True)
MODEL_DIR.mkdir(exist_ok=True, parents=True)

TRAIN_AUDIO_DIR = DATA_DIR / 'train_audio'
TRAIN_SOUNDSCAPES = DATA_DIR / 'train_soundscapes'
TEST_SOUNDSCAPES = DATA_DIR / 'test_soundscapes'
TRAIN_CSV = DATA_DIR / 'train.csv'
TRAIN_SS_LABELS_CSV = DATA_DIR / 'train_soundscapes_labels.csv'
TAXONOMY_CSV = DATA_DIR / 'taxonomy.csv'
SAMPLE_SUB_CSV = DATA_DIR / 'sample_submission.csv'

print(cfg)
print('Device:', DEVICE)
print('Chunk samples:', CHUNK_SAMPLES)


Config(data_dir='data\\birdclef-2026', cache_dir='mel_cache_v2', model_dir='models', model_weights_dir='/kaggle/input/datasets/wiktorwoniak/timm-weights/effnet_b0_ns_jft_in1k.pth', sr=32000, chunk_seconds=5, n_fft=1024, hop_length=512, n_mels=128, fmin=50, fmax=14000, top_db=80, max_files_per_species=40, val_frac=0.15, test_frac=0.15, split_seed=42, batch_size=24, epochs=4, lr=0.0003, weight_decay=0.0001, mixup_alpha=0.3, use_spec_augment=True, secondary_label_weight=0.25, label_smoothing=0.02, freeze_backbone_epochs=1, grad_clip=1.0, model_name='tf_efficientnet_b0.ns_jft_in1k', pretrained=True, in_chans=3, dropout=0.0, train_random_chunks=1, val_num_chunks=3, infer_num_chunks=12, max_audio_seconds_for_cache_probe=180, sample_strategy='sqrt_inv', num_workers=0, amp=True, experiment_name='birdclef_improved_v1')
Device: cpu
Chunk samples: 160000


In [3]:
taxonomy = pd.read_csv(TAXONOMY_CSV)
train_df = pd.read_csv(TRAIN_CSV)
train_ss_df = pd.read_csv(TRAIN_SS_LABELS_CSV) if TRAIN_SS_LABELS_CSV.exists() else None
sample_sub = pd.read_csv(SAMPLE_SUB_CSV) if SAMPLE_SUB_CSV.exists() else None

label_col_candidates = ['primary_label', 'primarylabel']
for c in label_col_candidates:
    if c in taxonomy.columns:
        tax_label_col = c
        break
else:
    raise ValueError('Could not find taxonomy label column')

for c in label_col_candidates:
    if c in train_df.columns:
        train_label_col = c
        break
else:
    raise ValueError('Could not find train label column')

filename_col = 'filename'
secondary_col = 'secondary_labels' if 'secondary_labels' in train_df.columns else 'secondarylabels'
rating_col = 'rating' if 'rating' in train_df.columns else None

species = taxonomy[tax_label_col].astype(str).tolist()
label2idx = {s: i for i, s in enumerate(species)}
num_classes = len(species)

train_df[train_label_col] = train_df[train_label_col].astype(str)
train_df = train_df[train_df[train_label_col].isin(label2idx)].copy().reset_index(drop=True)

print('taxonomy rows:', len(taxonomy))
print('train rows:', len(train_df))
print('num classes:', num_classes)
print('train soundscapes labels present:', train_ss_df is not None)
train_df.head()


taxonomy rows: 234
train rows: 35549
num classes: 234
train soundscapes labels present: True


,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat


In [4]:
if rating_col is not None:
    subset_df = (
        train_df
        .sort_values(rating_col, ascending=False)
        .groupby(train_label_col, group_keys=False)
        .head(cfg.max_files_per_species)
        .reset_index(drop=True)
    )
else:
    subset_df = (
        train_df
        .groupby(train_label_col, group_keys=False)
        .head(cfg.max_files_per_species)
        .reset_index(drop=True)
    )

rng = np.random.RandomState(cfg.split_seed)
train_idx, val_idx, test_idx = [], [], []
for sp, grp in subset_df.groupby(train_label_col):
    idx = grp.index.to_numpy().copy()
    rng.shuffle(idx)
    n = len(idx)
    if n < 3:
        train_idx.extend(idx.tolist())
        continue
    n_test = max(1, int(round(n * cfg.test_frac)))
    n_val = max(1, int(round(n * cfg.val_frac)))
    if n_test + n_val >= n:
        n_test, n_val = 1, 1
    test_idx.extend(idx[:n_test].tolist())
    val_idx.extend(idx[n_test:n_test + n_val].tolist())
    train_idx.extend(idx[n_test + n_val:].tolist())

train_subset = subset_df.loc[train_idx].reset_index(drop=True)
val_subset = subset_df.loc[val_idx].reset_index(drop=True)
test_subset = subset_df.loc[test_idx].reset_index(drop=True)

print('subset size:', len(subset_df))
print('train / val / test:', len(train_subset), len(val_subset), len(test_subset))
print('val species coverage:', val_subset[train_label_col].nunique(), '/', num_classes)
print('test species coverage:', test_subset[train_label_col].nunique(), '/', num_classes)


subset size: 6911
train / val / test: 4831 1040 1040
val species coverage: 199 / 234
test species coverage: 199 / 234


In [5]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=cfg.sr,
    n_fft=cfg.n_fft,
    hop_length=cfg.hop_length,
    n_mels=cfg.n_mels,
    f_min=cfg.fmin,
    f_max=cfg.fmax,
    power=2.0,
)
db_transform = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=cfg.top_db)


def load_audio(path: Path, sr: int = cfg.sr) -> np.ndarray:
    try:
        wav, file_sr = torchaudio.load(str(path))
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if file_sr != sr:
            wav = torchaudio.functional.resample(wav, file_sr, sr)
        return wav.squeeze(0).numpy().astype(np.float32)
    except Exception:
        y, _ = librosa.load(str(path), sr=sr, mono=True)
        return y.astype(np.float32)


def waveform_to_logmel(y: np.ndarray) -> np.ndarray:
    if len(y) < CHUNK_SAMPLES:
        y = np.pad(y, (0, CHUNK_SAMPLES - len(y)))
    elif len(y) > CHUNK_SAMPLES:
        y = y[:CHUNK_SAMPLES]
    t = torch.from_numpy(y).float().unsqueeze(0)
    mel = mel_transform(t)
    logmel = db_transform(mel).squeeze(0)
    logmel = (logmel - logmel.mean()) / (logmel.std() + 1e-6)
    return logmel.numpy().astype(np.float32)


def cache_base_path(rel_filename: str) -> Path:
    h = hashlib.md5(rel_filename.encode()).hexdigest()[:20]
    return CACHE_DIR / h


def metadata_path(rel_filename: str) -> Path:
    return cache_base_path(rel_filename).with_suffix('.json')


def chunks_npy_path(rel_filename: str) -> Path:
    return cache_base_path(rel_filename).with_suffix('.npy')


def compute_chunk_starts(num_samples: int, chunk_samples: int, hop_samples: int) -> List[int]:
    if num_samples <= chunk_samples:
        return [0]
    starts = list(range(0, num_samples - chunk_samples + 1, hop_samples))
    if starts[-1] != num_samples - chunk_samples:
        starts.append(num_samples - chunk_samples)
    return starts


def compute_and_cache_file(rel_filename: str, stride_seconds: float = 2.5) -> Path:
    meta_path = metadata_path(rel_filename)
    arr_path = chunks_npy_path(rel_filename)
    if meta_path.exists() and arr_path.exists():
        return arr_path

    src = TRAIN_AUDIO_DIR / rel_filename
    y = load_audio(src)
    hop = max(1, int(cfg.sr * stride_seconds))
    starts = compute_chunk_starts(len(y), CHUNK_SAMPLES, hop)
    specs = []
    energies = []
    for st in starts:
        chunk = y[st:st + CHUNK_SAMPLES]
        if len(chunk) < CHUNK_SAMPLES:
            chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)))
        specs.append(waveform_to_logmel(chunk))
        energies.append(float(np.mean(np.abs(chunk))))
    specs = np.stack(specs).astype(np.float32)
    np.save(arr_path, specs)
    meta = {
        'filename': rel_filename,
        'num_samples': int(len(y)),
        'num_chunks': int(len(specs)),
        'chunk_seconds': cfg.chunk_seconds,
        'stride_seconds': stride_seconds,
        'energies': energies,
        'shape': list(specs.shape),
    }
    meta_path.write_text(json.dumps(meta))
    return arr_path


In [6]:
from concurrent.futures import ThreadPoolExecutor

all_files = pd.concat([train_subset, val_subset, test_subset])[filename_col].drop_duplicates().tolist()
missing = [f for f in all_files if not chunks_npy_path(f).exists()]
print('files in split union:', len(all_files))
print('to cache:', len(missing))

if missing:
    workers = max(1, min(4, os.cpu_count() or 2))
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for i, _ in enumerate(ex.map(compute_and_cache_file, missing), 1):
            if i % 100 == 0 or i == len(missing):
                print(f'cached {i}/{len(missing)}')

probe_file = all_files[0]
probe = np.load(chunks_npy_path(probe_file), mmap_mode='r')
print('probe file:', probe_file)
print('cached chunk tensor shape:', probe.shape)


files in split union: 6911
to cache: 0
probe file: 1161364/iNat842139.ogg
cached chunk tensor shape: (4, 128, 313)


In [7]:
class BirdClefCachedDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        train: bool,
        num_eval_chunks: int = 1,
    ):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.num_eval_chunks = num_eval_chunks
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=20)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=32)

    def __len__(self):
        return len(self.df)

    def parse_secondary(self, value) -> List[str]:
        if not isinstance(value, str) or value.strip() in {'', '[]', 'nan'}:
            return []
        try:
            parsed = json.loads(value.replace("'", '"')) if value.strip().startswith('[') else []
            return [str(v) for v in parsed]
        except Exception:
            return []

    def make_target(self, row) -> torch.Tensor:
        t = torch.zeros(num_classes, dtype=torch.float32)
        p = str(row[train_label_col])
        if p in label2idx:
            t[label2idx[p]] = 1.0 - cfg.label_smoothing
        for s in self.parse_secondary(row.get(secondary_col, '[]')):
            if s in label2idx and t[label2idx[s]] == 0:
                t[label2idx[s]] = cfg.secondary_label_weight
        return t

    def choose_chunk_indices(self, n_chunks: int) -> List[int]:
        if n_chunks <= 1:
            return [0]
        if self.train:
            return [np.random.randint(0, n_chunks)]
        if self.num_eval_chunks >= n_chunks:
            return list(range(n_chunks))
        xs = np.linspace(0, n_chunks - 1, self.num_eval_chunks)
        return sorted(set(int(round(x)) for x in xs))

    def postprocess_spec(self, x: torch.Tensor) -> torch.Tensor:
        if self.train and cfg.use_spec_augment:
            x = self.freq_mask(x)
            x = self.time_mask(x)
        if cfg.in_chans == 3:
            x = x.repeat(3, 1, 1)
        return x

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        specs = np.load(chunks_npy_path(row[filename_col]), mmap_mode='r')
        ids = self.choose_chunk_indices(len(specs))
        selected = []
        for cid in ids:
            x = torch.from_numpy(np.array(specs[cid], copy=True)).unsqueeze(0)
            x = self.postprocess_spec(x)
            selected.append(x)
        x = selected[0] if len(selected) == 1 else torch.stack(selected, dim=0)
        y = self.make_target(row)
        return x, y, str(row[filename_col])


def eval_collate(batch):
    xs, ys, fns = zip(*batch)
    return list(xs), torch.stack(ys), list(fns)


In [8]:
train_ds = BirdClefCachedDataset(train_subset, train=True, num_eval_chunks=1)
val_ds = BirdClefCachedDataset(val_subset, train=False, num_eval_chunks=cfg.val_num_chunks)
test_ds = BirdClefCachedDataset(test_subset, train=False, num_eval_chunks=cfg.val_num_chunks)

train_labels = train_subset[train_label_col].map(label2idx).values
class_counts = np.bincount(train_labels, minlength=num_classes)
if cfg.sample_strategy == 'sqrt_inv':
    class_weights = 1.0 / np.sqrt(np.maximum(class_counts, 1))
else:
    class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(sample_weights.tolist(), num_samples=len(train_labels), replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    sampler=sampler,
    num_workers=cfg.num_workers,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=max(1, cfg.batch_size // 2),
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=eval_collate,
)
test_loader = DataLoader(
    test_ds,
    batch_size=max(1, cfg.batch_size // 2),
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=eval_collate,
)

xb, yb, fnb = next(iter(train_loader))
print('train batch x:', xb.shape)
print('train batch y:', yb.shape)
print('sample file:', fnb[0])


train batch x: torch.Size([24, 3, 128, 313])
train batch y: torch.Size([24, 234])
sample file: 25092/iNat854292.ogg


In [21]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        xs_pos = torch.sigmoid(logits)
        xs_neg = 1.0 - xs_pos
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)
        loss_pos = targets * torch.log(xs_pos.clamp(min=self.eps))
        loss_neg = (1 - targets) * torch.log(xs_neg.clamp(min=self.eps))
        loss = loss_pos + loss_neg
        pt = xs_pos * targets + xs_neg * (1 - targets)
        gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
        one_sided_w = torch.pow(1 - pt, gamma)
        return -(one_sided_w * loss).mean()


def build_model() -> nn.Module:
    model = timm.create_model(
        cfg.model_name,
        pretrained=cfg.pretrained,
        num_classes=num_classes,
        in_chans=cfg.in_chans,
        drop_rate=cfg.dropout,
    )
    return model

model = build_model().to(DEVICE)
print('Model:', cfg.model_name)

state = torch.load(
    cfg.model_weights_dir,
    map_location=DEVICE
)

model.load_state_dict(state)


Model: tf_efficientnet_b0.ns_jft_in1k


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/wiktorwoniak/timm-weights/effnet_b0_ns_jft_in1k.pth'

In [11]:
def freeze_backbone(model: nn.Module, freeze: bool = True):
    head_keywords = ['classifier', 'fc', 'head']
    for name, p in model.named_parameters():
        is_head = any(k in name.lower() for k in head_keywords)
        p.requires_grad = (not freeze) or is_head

criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.epochs * max(1, len(train_loader))))
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and DEVICE.type == 'cuda'))

if cfg.freeze_backbone_epochs > 0:
    freeze_backbone(model, True)


In [12]:
class ParticipantVisibleError(Exception):
    pass


def competition_score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    solution = solution.copy()
    submission = submission.copy()
    del solution[row_id_column_name]
    del submission[row_id_column_name]
    if not all(pd.api.types.is_numeric_dtype(submission[c]) for c in submission.columns):
        bad = {c: submission[c].dtype for c in submission.columns if not pd.api.types.is_numeric_dtype(submission[c])}
        raise ParticipantVisibleError(f'Invalid submission dtypes: {bad}')
    solution_sums = solution.sum(axis=0)
    scored_columns = list(solution_sums[solution_sums > 0].index.values)
    assert len(scored_columns) > 0
    return float(roc_auc_score(solution[scored_columns].values, submission[scored_columns].values, average='macro'))


def score_from_arrays(targets: np.ndarray, probs: np.ndarray) -> float:
    sol = pd.DataFrame((targets > 0).astype(int), columns=species)
    sub = pd.DataFrame(probs, columns=species)
    sol.insert(0, 'row_id', np.arange(len(sol)))
    sub.insert(0, 'row_id', np.arange(len(sub)))
    return competition_score(sol, sub, 'row_id')


def mixup_batch(x, y, alpha: float):
    if alpha <= 0:
        return x, y
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x = lam * x + (1 - lam) * x[perm]
    y = torch.clamp(lam * y + (1 - lam) * y[perm], 0.0, 1.0)
    return x, y


def forward_eval_batch(model: nn.Module, batch_x_list: List[torch.Tensor]) -> torch.Tensor:
    batch_probs = []
    for x in batch_x_list:
        if x.ndim == 3:
            x = x.unsqueeze(0)
        elif x.ndim == 4:
            pass
        else:
            raise ValueError(f'Unexpected eval tensor shape: {x.shape}')
        x = x.to(DEVICE)
        n_views = x.shape[0]
        logits = model(x)
        probs = torch.sigmoid(logits)
        probs = probs.mean(dim=0)
        batch_probs.append(probs)
    return torch.stack(batch_probs, dim=0)


def train_one_epoch(epoch: int):
    model.train()
    losses = []
    if epoch == cfg.freeze_backbone_epochs + 1:
        freeze_backbone(model, False)
    for xb, yb, _ in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        xb, yb = mixup_batch(xb, yb, cfg.mixup_alpha)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(cfg.amp and DEVICE.type == 'cuda')):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        if cfg.grad_clip is not None and cfg.grad_clip > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(loss.item())
    return float(np.mean(losses)) if losses else float('nan')

@torch.no_grad()
def evaluate(loader):
    model.eval()
    all_probs, all_targets = [], []
    losses = []
    for x_list, yb, _ in loader:
        yb = yb.to(DEVICE)
        probs = forward_eval_batch(model, x_list)
        probs = probs.clamp(1e-6, 1 - 1e-6)
        logits_proxy = torch.logit(probs)
        loss = criterion(logits_proxy, yb)
        losses.append(loss.item())
        all_probs.append(probs.cpu().numpy())
        all_targets.append(yb.cpu().numpy())
    probs = np.concatenate(all_probs, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    try:
        comp_auc = score_from_arrays(targets, probs)
    except Exception:
        comp_auc = float('nan')
    present = (targets > 0).any(axis=0)
    if present.any():
        try:
            macro_map = float(average_precision_score((targets[:, present] > 0).astype(int), probs[:, present], average='macro'))
        except Exception:
            macro_map = float('nan')
    else:
        macro_map = float('nan')
    top1 = probs.argmax(axis=1)
    top1_acc = float(np.mean([targets[i, j] > 0 for i, j in enumerate(top1)]))
    return {
        'loss': float(np.mean(losses)) if losses else float('nan'),
        'comp_auc': float(comp_auc),
        'macro_map': float(macro_map),
        'top1_acc': float(top1_acc),
        'probs': probs,
        'targets': targets,
    }


In [14]:
history = []
best_score = -1.0
best_path = MODEL_DIR / f'{cfg.experiment_name}_{cfg.model_name.replace("/", "_")}.pt'

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    train_loss = train_one_epoch(epoch)
    val_metrics = evaluate(val_loader)
    elapsed = time.time() - t0
    lr_now = optimizer.param_groups[0]['lr']

    row = {
        'epoch': epoch,
        'lr': lr_now,
        'train_loss': train_loss,
        'val_loss': val_metrics['loss'],
        'val_comp_auc': val_metrics['comp_auc'],
        'val_macro_map': val_metrics['macro_map'],
        'val_top1_acc': val_metrics['top1_acc'],
        'seconds': elapsed,
    }
    history.append(row)
    print(row)

    score = val_metrics['comp_auc']
    if score == score and score > best_score:
        best_score = score
        payload = {
            'model_state': model.state_dict(),
            'config': asdict(cfg),
            'species': species,
            'best_score': best_score,
            'history': history,
        }
        torch.save(payload, best_path)
        print('saved best checkpoint to', best_path)

history_df = pd.DataFrame(history)
history_df


{'epoch': 1, 'lr': 0.0002926584774442728, 'train_loss': 0.005814254954017105, 'val_loss': 0.00459391289208641, 'val_comp_auc': 0.5834026265652431, 'val_macro_map': 0.02105453022521195, 'val_top1_acc': 0.014423076923076924, 'seconds': 421.927693605423}
saved best checkpoint to models\birdclef_improved_v1_tf_efficientnet_b0.ns_jft_in1k.pt
{'epoch': 2, 'lr': 0.0002713525491562418, 'train_loss': 0.0032153984923286374, 'val_loss': 0.0037787528093047866, 'val_comp_auc': 0.7970615910786212, 'val_macro_map': 0.11317029578104541, 'val_top1_acc': 0.11923076923076924, 'seconds': 924.2758119106293}
saved best checkpoint to models\birdclef_improved_v1_tf_efficientnet_b0.ns_jft_in1k.pt
{'epoch': 3, 'lr': 0.0002381677878438708, 'train_loss': 0.002672310129146263, 'val_loss': 0.0032903691914317936, 'val_comp_auc': 0.8774988987521489, 'val_macro_map': 0.2534495119677125, 'val_top1_acc': 0.24134615384615385, 'seconds': 920.5319855213165}
saved best checkpoint to models\birdclef_improved_v1_tf_efficientn

,epoch,lr,train_loss,val_loss,val_comp_auc,val_macro_map,val_top1_acc,seconds
0,1,0.000293,0.005814,0.004594,0.583403,0.021055,0.014423,421.927694
1,2,0.000271,0.003215,0.003779,0.797062,0.113170,0.119231,924.275812
2,3,0.000238,0.002672,0.003290,0.877499,0.253450,0.241346,920.531986
3,4,0.000196,0.002292,0.002932,0.906524,0.353708,0.367308,801.391898
4,5,0.000150,0.002079,0.002564,0.921536,0.412526,0.449038,782.257054


In [16]:
if best_path.exists():
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    print('Loaded best checkpoint with val_comp_auc =', ckpt['best_score'])

# test_metrics = evaluate(test_loader)
# test_metrics_summary = {
#     'test_loss': test_metrics['loss'],
#     'test_comp_auc': test_metrics['comp_auc'],
#     'test_macro_map': test_metrics['macro_map'],
#     'test_top1_acc': test_metrics['top1_acc'],
# }
# test_metrics_summary


Loaded best checkpoint with val_comp_auc = 0.9215357361606556


In [17]:
def predict_soundscape(path: Path, num_chunks: int = None, aggregate: str = 'mean'):
    if num_chunks is None:
        num_chunks = cfg.infer_num_chunks
    y = load_audio(path)
    total = len(y)
    rows = []
    model.eval()
    with torch.no_grad():
        for end_sec in range(cfg.chunk_seconds, 61, cfg.chunk_seconds):
            start = (end_sec - cfg.chunk_seconds) * cfg.sr
            end = end_sec * cfg.sr
            seg = y[start:end] if end <= total else np.pad(y[start:total], (0, max(0, end - total)))
            if len(seg) < CHUNK_SAMPLES:
                seg = np.pad(seg, (0, CHUNK_SAMPLES - len(seg)))

            views = [waveform_to_logmel(seg)]
            if num_chunks > 1:
                shifts = np.linspace(-0.75, 0.75, num_chunks)
                for s in shifts[1:]:
                    shift = int(round(s * cfg.sr))
                    if shift >= 0:
                        shifted = np.pad(seg[shift:], (0, shift))
                    else:
                        shifted = np.pad(seg[:shift], (-shift, 0))
                    views.append(waveform_to_logmel(shifted[:CHUNK_SAMPLES]))

            x = torch.from_numpy(np.stack(views)).unsqueeze(1)
            if cfg.in_chans == 3:
                x = x.repeat(1, 3, 1, 1)
            x = x.to(DEVICE)
            probs = torch.sigmoid(model(x))
            probs = probs.max(dim=0).values if aggregate == 'max' else probs.mean(dim=0)
            rows.append((f'{path.name[:-4]}_{end_sec}', probs.cpu().numpy()))
    return rows

if TEST_SOUNDSCAPES.exists() and any(TEST_SOUNDSCAPES.iterdir()):
    out_rows = []
    for p in sorted(TEST_SOUNDSCAPES.glob('*.ogg')):
        print('predicting', p.name)
        for row_id, probs in predict_soundscape(p, aggregate='mean'):
            out_rows.append([row_id] + probs.tolist())
    submission = pd.DataFrame(out_rows, columns=['row_id'] + species)
    submission.to_csv('submission.csv', index=False)
    print('wrote submission.csv', submission.shape)
else:
    print('test_soundscapes is empty or unavailable; submission step skipped locally.')


wrote submission.csv (0, 235)


## Notes

Suggested next experiments:

- Run the same notebook with 3 different `split_seed` values and average checkpoints.
- Compare `tf_efficientnet_b0` against `mobilenetv3_small_100` for CPU speed.
- Try `in_chans=1` with a custom first conv adaptation if you move away from ImageNet-style transfer.
- Increase `val_num_chunks` and `infer_num_chunks` if runtime permits.
- Add train soundscape supervision as an additional dataset once the baseline is stable.
